# NLP – Tutorial 1 Lab: German Text Corpus
**Group 2 – German** · spaCy model `de_core_news_md`
**Members:** Blevis Allushi · Darsi Ramaj · Geri Belba

All logic lives in the `sprache` package (`src/sprache/`); this notebook only calls it and discusses the results. The corpus is **frozen** in `corpus/` + `sources.csv`, so a run needs no internet. Set `REBUILD = True` in section 1 to download it again.

**Run:** Kernel → *Restart Kernel and Run All Cells* (or `python -m sprache notebook`). Setup: see `README.md`.

| Part | Owner | Sections |
|---|---|---|
| Setup, pipeline demo, displaCy, corpus collection, UTF-8 check, three sentences | **Blevis** | 0, 1, 5 |
| Sources, licences, quality problems, discussion | **Geri** | 2, 4, 7 |
| Corpus statistics, quality problems, German-specific behaviour, table row | **Darsi** | 3, 4, 6 |

## 0 · Setup and version printout  *(Blevis)*

In [1]:
import sys
from pathlib import Path

try:                                   # installed with `pip install -e .`
    import sprache
except ImportError:                    # or run straight from the repo
    sys.path.insert(0, str(Path('../src').resolve()))
    import sprache

import pandas as pd
import spacy
from IPython.display import HTML, Markdown, display
from spacy import displacy

from sprache import analysis, clean, config, corpus, quality, stats

pd.set_option('display.max_colwidth', 120)
print('Python :', sys.version.split()[0])
print('spaCy  :', spacy.__version__)
nlp = analysis.load_model()            # clear error message if the model is missing
print('Model  :', f"{nlp.meta['lang']}_{nlp.meta['name']} {nlp.meta['version']}")
print('Pipes  :', nlp.pipe_names)
print('sprache:', sprache.__version__)

Python : 3.11.17
spaCy  : 3.8.18


Model  : de_core_news_md 3.8.0
Pipes  : ['tok2vec', 'tagger', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']
sprache: 3.0.0


One sentence through the full pipeline (tokenizer, tagger, morphologizer, parser, lemmatizer, NER):

In [2]:
demo = nlp(config.DEMO_SENTENCE)
display(analysis.token_table(demo, morph=True))
display(analysis.entity_table(demo))

,token,lemma,pos,tag,morph,dep,head,ent
i,,,,,,,,
0,Johann,Johann,PROPN,NE,Case=Nom|Gender=Masc|Number=Sing,pnc,Goethe,B-PER
1,Wolfgang,Wolfgang,PROPN,NE,Case=Nom|Gender=Masc|Number=Sing,pnc,Goethe,I-PER
2,von,von,PROPN,NE,,pnc,Goethe,I-PER
3,Goethe,Goethe,PROPN,NE,Case=Nom|Gender=Masc|Number=Sing,sb,wurde,I-PER
4,wurde,werden,AUX,VAFIN,Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin,ROOT,wurde,
5,1749,1749,NUM,CARD,,mo,geboren,
6,in,in,ADP,APPR,,mo,geboren,
7,Frankfurt,Frankfurt,PROPN,NE,Case=Dat|Gender=Neut|Number=Sing,nk,in,B-LOC
8,am,an,ADP,APPRART,Case=Dat|Gender=Masc|Number=Sing,mnr,Frankfurt,I-LOC


,entity,label,start,end
0,Johann Wolfgang von Goethe,PER,0,4
1,Frankfurt am Main,LOC,7,10


In [3]:
displacy.render(demo, style='dep', jupyter=True, options={'distance': 95})
displacy.render(demo, style='ent', jupyter=True)
analysis.save_displacy(demo, 'demo');   # figures/demo_dep.svg, figures/demo_ent.html

## 1 · Collecting the texts  *(Blevis)*
Text type: **introductions of German Wikipedia articles** (CC BY-SA 4.0, reuse allowed with attribution), fetched through the official MediaWiki API, not by scraping.

How `corpus.build()` works:
- Candidate titles come from `data/titles.txt`, grouped into 8 topics (cities, nature, people, companies, history, culture, science, language/society). They are taken **round-robin**, so the corpus stays balanced across topics.
- Titles are requested **20 per API call** with retry and back-off on HTTP 429. Version 1 used one call per title and lost 14 of its 34 titles to rate limiting (HTTP 429).
- Each intro is cut at a sentence boundary to at most 300 words. The splitter does not cut after ordinals (*19. Jahrhundert*) or abbreviations (*z. B.*). Intros under 100 words, disambiguation pages, missing pages and redirect duplicates are skipped and logged in `data/skipped.csv`.
- Every text gets a permanent link to the exact revision used (`oldid`), so the corpus is reproducible even after the articles change.

In [4]:
REBUILD = False          # True -> download the corpus again (needs internet, ~10 s)

if REBUILD or not config.SOURCES_CSV.exists():
    result = corpus.build()
    clean.clean_corpus()
    print(f'Built {len(result.sources)} texts, skipped {len(result.skipped)} titles.')

sources = pd.DataFrame(corpus.read_sources())
sources['words'] = sources['words'].astype(int)
raw = corpus.load(config.CORPUS_RAW)
print(f"{len(raw)} texts in {config.CORPUS_RAW.name}/, {sources.words.sum()} words "
      f"(min {sources.words.min()}, median {int(sources.words.median())}, max {sources.words.max()})")
display(sources.groupby('topic').agg(texts=('file', 'count'), words=('words', 'sum'),
                                     examples=('title', lambda t: ', '.join(t[:4]))))

100 texts in corpus/, 21059 words (min 102, median 210, max 300)


,texts,words,examples
topic,,,
Geschichte und Politik,12,3183,"Heiliges Römisches Reich, Reformation, Dreißigjähriger Krieg, Deutsches Kaiserreich"
Kultur und Alltag,11,2008,"Oktoberfest, Brandenburger Tor, Kölner Dom, Weihnachtsmarkt"
Natur und Geografie,13,2249,"Rhein, Donau, Elbe, Bodensee"
Personen,16,3736,"Johann Wolfgang von Goethe, Albert Einstein, Ludwig van Beethoven, Johann Sebastian Bach"
Sprache und Gesellschaft,11,2116,"Deutsche Sprache, Dialekt, Umlaut, Komposition (Grammatik)"
Städte,15,3750,"Berlin, München, Hamburg, Köln"
Unternehmen und Organisationen,8,1226,"Siemens, BMW, Deutsche Bahn, Deutscher Fußball-Bund"
Wissenschaft und Technik,14,2791,"Automobil, Buchdruck, Relativitätstheorie, Photosynthese"


In [5]:
skipped = pd.read_csv(config.SKIPPED_CSV)
skipped['reason_type'] = skipped.reason.str.replace(r'\d+', 'N', regex=True)
display(skipped.reason_type.value_counts().rename('titles skipped'))

reason_type
only N words         34
MAX_TEXTS reached    13
Name: titles skipped, dtype: int64

**UTF-8 check:** re-open every file as UTF-8, look for mojibake (e.g. `Ã¤` for *ä*) and confirm the German special characters survived.

In [6]:
from collections import Counter
chars = Counter(c for t in raw.values() for c in t if c in 'äöüÄÖÜß')
moji = quality.pattern_report(raw).set_index('check').loc['mojibake (broken UTF-8)']
print('Files with mojibake      :', moji.files or 'none')
print('Special characters found :', dict(chars.most_common()))
sample = max((s for t in raw.values() for s in corpus.split_sentences(t) if len(s.split()) <= 25),
             key=lambda s: len(set(s) & set('äöüÄÖÜß')))   # short sentence with the most distinct special chars
print('Sample sentence          :', sample)

Files with mojibake      : none
Special characters found : {'ä': 801, 'ü': 651, 'ö': 425, 'ß': 267, 'Ö': 32, 'Ü': 22, 'Ä': 13}
Sample sentence          : Überregionale Bekanntheit genießt Düsseldorf durch seine Altstadt („längste Theke der Welt“), seinen Einkaufsboulevard Königsallee („Kö“), seinen Standort für Kleidermode und den Düsseldorfer Karneval.


## 2 · Sources and licences  *(Geri)*

In [7]:
lines = [f"{i}. **{r.title}** ({r.topic}) – <{r.url}> – revision [{r.revision_id}]({r.permalink}) – "
         f"last modified {r.last_modified}, retrieved {r.retrieved} – {r.licence} – `{r.file}`"
         for i, r in enumerate(sources.itertuples(), 1)]
display(Markdown(
    f"All {len(sources)} texts are introductions of German Wikipedia articles, licensed "
    f"**[{config.LICENCE}]({config.LICENCE_URL})**. Authors: the Wikipedia contributors listed "
    "in each article's history. We only shortened the texts (first ≤ 300 words of the intro) and, in "
    "`corpus_clean/`, removed artefacts (section 4). Under share-alike, the derived corpus keeps the same licence.\n\n"
    + "\n".join(lines)))

All 100 texts are introductions of German Wikipedia articles, licensed **[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)**. Authors: the Wikipedia contributors listed in each article's history. We only shortened the texts (first ≤ 300 words of the intro) and, in `corpus_clean/`, removed artefacts (section 4). Under share-alike, the derived corpus keeps the same licence.

1. **Berlin** (Städte) – <https://de.wikipedia.org/wiki/Berlin> – revision [271960099](https://de.wikipedia.org/w/index.php?oldid=271960099) – last modified 2026-11-07, retrieved 2026-11-15 – CC BY-SA 4.0 – `text001.txt`
2. **Rhein** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Rhein> – revision [271927846](https://de.wikipedia.org/w/index.php?oldid=271927846) – last modified 2026-11-06, retrieved 2026-11-15 – CC BY-SA 4.0 – `text002.txt`
3. **Johann Wolfgang von Goethe** (Personen) – <https://de.wikipedia.org/wiki/Johann_Wolfgang_von_Goethe> – revision [272180154](https://de.wikipedia.org/w/index.php?oldid=272180154) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text003.txt`
4. **Heiliges Römisches Reich** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Heiliges_R%C3%B6misches_Reich> – revision [271723860](https://de.wikipedia.org/w/index.php?oldid=271723860) – last modified 2026-10-29, retrieved 2026-11-15 – CC BY-SA 4.0 – `text004.txt`
5. **Oktoberfest** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Oktoberfest> – revision [272148031](https://de.wikipedia.org/w/index.php?oldid=272148031) – last modified 2026-11-14, retrieved 2026-11-15 – CC BY-SA 4.0 – `text005.txt`
6. **Automobil** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Automobil> – revision [269830996](https://de.wikipedia.org/w/index.php?oldid=269830996) – last modified 2026-08-20, retrieved 2026-11-15 – CC BY-SA 4.0 – `text006.txt`
7. **Deutsche Sprache** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Deutsche_Sprache> – revision [272143394](https://de.wikipedia.org/w/index.php?oldid=272143394) – last modified 2026-11-13, retrieved 2026-11-15 – CC BY-SA 4.0 – `text007.txt`
8. **München** (Städte) – <https://de.wikipedia.org/wiki/M%C3%BCnchen> – revision [272021669](https://de.wikipedia.org/w/index.php?oldid=272021669) – last modified 2026-11-09, retrieved 2026-11-15 – CC BY-SA 4.0 – `text008.txt`
9. **Donau** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Donau> – revision [272037802](https://de.wikipedia.org/w/index.php?oldid=272037802) – last modified 2026-11-10, retrieved 2026-11-15 – CC BY-SA 4.0 – `text009.txt`
10. **Siemens** (Unternehmen und Organisationen) – <https://de.wikipedia.org/wiki/Siemens> – revision [272082266](https://de.wikipedia.org/w/index.php?oldid=272082266) – last modified 2026-11-11, retrieved 2026-11-15 – CC BY-SA 4.0 – `text010.txt`
11. **Reformation** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Reformation> – revision [271838880](https://de.wikipedia.org/w/index.php?oldid=271838880) – last modified 2026-11-02, retrieved 2026-11-15 – CC BY-SA 4.0 – `text011.txt`
12. **Brandenburger Tor** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Brandenburger_Tor> – revision [272011591](https://de.wikipedia.org/w/index.php?oldid=272011591) – last modified 2026-11-09, retrieved 2026-11-15 – CC BY-SA 4.0 – `text012.txt`
13. **Buchdruck** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Buchdruck> – revision [269019864](https://de.wikipedia.org/w/index.php?oldid=269019864) – last modified 2026-07-22, retrieved 2026-11-15 – CC BY-SA 4.0 – `text013.txt`
14. **Dialekt** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Dialekt> – revision [269758746](https://de.wikipedia.org/w/index.php?oldid=269758746) – last modified 2026-08-18, retrieved 2026-11-15 – CC BY-SA 4.0 – `text014.txt`
15. **Hamburg** (Städte) – <https://de.wikipedia.org/wiki/Hamburg> – revision [271777581](https://de.wikipedia.org/w/index.php?oldid=271777581) – last modified 2026-10-31, retrieved 2026-11-15 – CC BY-SA 4.0 – `text015.txt`
16. **Elbe** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Elbe> – revision [271988957](https://de.wikipedia.org/w/index.php?oldid=271988957) – last modified 2026-11-08, retrieved 2026-11-15 – CC BY-SA 4.0 – `text016.txt`
17. **Albert Einstein** (Personen) – <https://de.wikipedia.org/wiki/Albert_Einstein> – revision [271734114](https://de.wikipedia.org/w/index.php?oldid=271734114) – last modified 2026-10-29, retrieved 2026-11-15 – CC BY-SA 4.0 – `text017.txt`
18. **BMW** (Unternehmen und Organisationen) – <https://de.wikipedia.org/wiki/BMW> – revision [270991039](https://de.wikipedia.org/w/index.php?oldid=270991039) – last modified 2026-10-03, retrieved 2026-11-15 – CC BY-SA 4.0 – `text018.txt`
19. **Dreißigjähriger Krieg** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Drei%C3%9Figj%C3%A4hriger_Krieg> – revision [270660817](https://de.wikipedia.org/w/index.php?oldid=270660817) – last modified 2026-09-20, retrieved 2026-11-15 – CC BY-SA 4.0 – `text019.txt`
20. **Kölner Dom** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/K%C3%B6lner_Dom> – revision [271917040](https://de.wikipedia.org/w/index.php?oldid=271917040) – last modified 2026-11-05, retrieved 2026-11-15 – CC BY-SA 4.0 – `text020.txt`
21. **Relativitätstheorie** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Relativit%C3%A4tstheorie> – revision [267053488](https://de.wikipedia.org/w/index.php?oldid=267053488) – last modified 2026-05-14, retrieved 2026-11-15 – CC BY-SA 4.0 – `text021.txt`
22. **Köln** (Städte) – <https://de.wikipedia.org/wiki/K%C3%B6ln> – revision [272037049](https://de.wikipedia.org/w/index.php?oldid=272037049) – last modified 2026-11-10, retrieved 2026-11-15 – CC BY-SA 4.0 – `text022.txt`
23. **Bodensee** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Bodensee> – revision [272064154](https://de.wikipedia.org/w/index.php?oldid=272064154) – last modified 2026-11-11, retrieved 2026-11-15 – CC BY-SA 4.0 – `text023.txt`
24. **Ludwig van Beethoven** (Personen) – <https://de.wikipedia.org/wiki/Ludwig_van_Beethoven> – revision [271347466](https://de.wikipedia.org/w/index.php?oldid=271347466) – last modified 2026-10-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text024.txt`
25. **Deutsches Kaiserreich** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Deutsches_Kaiserreich> – revision [272192289](https://de.wikipedia.org/w/index.php?oldid=272192289) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text025.txt`
26. **Umlaut** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Umlaut> – revision [270886167](https://de.wikipedia.org/w/index.php?oldid=270886167) – last modified 2026-09-29, retrieved 2026-11-15 – CC BY-SA 4.0 – `text026.txt`
27. **Frankfurt am Main** (Städte) – <https://de.wikipedia.org/wiki/Frankfurt_am_Main> – revision [271827027](https://de.wikipedia.org/w/index.php?oldid=271827027) – last modified 2026-11-02, retrieved 2026-11-15 – CC BY-SA 4.0 – `text027.txt`
28. **Johann Sebastian Bach** (Personen) – <https://de.wikipedia.org/wiki/Johann_Sebastian_Bach> – revision [272173326](https://de.wikipedia.org/w/index.php?oldid=272173326) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text028.txt`
29. **Weimarer Republik** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Weimarer_Republik> – revision [271390289](https://de.wikipedia.org/w/index.php?oldid=271390289) – last modified 2026-10-16, retrieved 2026-11-15 – CC BY-SA 4.0 – `text029.txt`
30. **Weihnachtsmarkt** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Weihnachtsmarkt> – revision [268306505](https://de.wikipedia.org/w/index.php?oldid=268306505) – last modified 2026-06-28, retrieved 2026-11-15 – CC BY-SA 4.0 – `text030.txt`
31. **Photosynthese** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Photosynthese> – revision [271875810](https://de.wikipedia.org/w/index.php?oldid=271875810) – last modified 2026-11-04, retrieved 2026-11-15 – CC BY-SA 4.0 – `text031.txt`
32. **Komposition (Grammatik)** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Komposition_(Grammatik)> – revision [272090759](https://de.wikipedia.org/w/index.php?oldid=272090759) – last modified 2026-11-12, retrieved 2026-11-15 – CC BY-SA 4.0 – `text032.txt`
33. **Stuttgart** (Städte) – <https://de.wikipedia.org/wiki/Stuttgart> – revision [271864779](https://de.wikipedia.org/w/index.php?oldid=271864779) – last modified 2026-11-03, retrieved 2026-11-15 – CC BY-SA 4.0 – `text033.txt`
34. **Zugspitze** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Zugspitze> – revision [271266791](https://de.wikipedia.org/w/index.php?oldid=271266791) – last modified 2026-10-12, retrieved 2026-11-15 – CC BY-SA 4.0 – `text034.txt`
35. **Wolfgang Amadeus Mozart** (Personen) – <https://de.wikipedia.org/wiki/Wolfgang_Amadeus_Mozart> – revision [271415616](https://de.wikipedia.org/w/index.php?oldid=271415616) – last modified 2026-10-17, retrieved 2026-11-15 – CC BY-SA 4.0 – `text035.txt`
36. **Deutsche Bahn** (Unternehmen und Organisationen) – <https://de.wikipedia.org/wiki/Deutsche_Bahn> – revision [272191291](https://de.wikipedia.org/w/index.php?oldid=272191291) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text036.txt`
37. **Berliner Mauer** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Berliner_Mauer> – revision [271989391](https://de.wikipedia.org/w/index.php?oldid=271989391) – last modified 2026-11-08, retrieved 2026-11-15 – CC BY-SA 4.0 – `text037.txt`
38. **Karneval, Fastnacht und Fasching** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Karneval,_Fastnacht_und_Fasching> – revision [269000865](https://de.wikipedia.org/w/index.php?oldid=269000865) – last modified 2026-07-21, retrieved 2026-11-15 – CC BY-SA 4.0 – `text038.txt`
39. **Klimawandel** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Klimawandel> – revision [269977423](https://de.wikipedia.org/w/index.php?oldid=269977423) – last modified 2026-08-25, retrieved 2026-11-15 – CC BY-SA 4.0 – `text039.txt`
40. **Duden** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Duden> – revision [271395961](https://de.wikipedia.org/w/index.php?oldid=271395961) – last modified 2026-10-17, retrieved 2026-11-15 – CC BY-SA 4.0 – `text040.txt`
41. **Düsseldorf** (Städte) – <https://de.wikipedia.org/wiki/D%C3%BCsseldorf> – revision [272160924](https://de.wikipedia.org/w/index.php?oldid=272160924) – last modified 2026-11-14, retrieved 2026-11-15 – CC BY-SA 4.0 – `text041.txt`
42. **Harz (Mittelgebirge)** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Harz_(Mittelgebirge)> – revision [269751929](https://de.wikipedia.org/w/index.php?oldid=269751929) – last modified 2026-08-17, retrieved 2026-11-15 – CC BY-SA 4.0 – `text042.txt`
43. **Immanuel Kant** (Personen) – <https://de.wikipedia.org/wiki/Immanuel_Kant> – revision [271888725](https://de.wikipedia.org/w/index.php?oldid=271888725) – last modified 2026-11-04, retrieved 2026-11-15 – CC BY-SA 4.0 – `text043.txt`
44. **Deutsche Wiedervereinigung** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Deutsche_Wiedervereinigung> – revision [271717099](https://de.wikipedia.org/w/index.php?oldid=271717099) – last modified 2026-10-29, retrieved 2026-11-15 – CC BY-SA 4.0 – `text044.txt`
45. **Brezel** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Brezel> – revision [272099979](https://de.wikipedia.org/w/index.php?oldid=272099979) – last modified 2026-11-12, retrieved 2026-11-15 – CC BY-SA 4.0 – `text045.txt`
46. **Elektrizität** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Elektrizit%C3%A4t> – revision [269413862](https://de.wikipedia.org/w/index.php?oldid=269413862) – last modified 2026-08-06, retrieved 2026-11-15 – CC BY-SA 4.0 – `text046.txt`
47. **Leipzig** (Städte) – <https://de.wikipedia.org/wiki/Leipzig> – revision [272126940](https://de.wikipedia.org/w/index.php?oldid=272126940) – last modified 2026-11-13, retrieved 2026-11-15 – CC BY-SA 4.0 – `text047.txt`
48. **Rügen** (Natur und Geografie) – <https://de.wikipedia.org/wiki/R%C3%BCgen> – revision [270927397](https://de.wikipedia.org/w/index.php?oldid=270927397) – last modified 2026-09-30, retrieved 2026-11-15 – CC BY-SA 4.0 – `text048.txt`
49. **Karl Marx** (Personen) – <https://de.wikipedia.org/wiki/Karl_Marx> – revision [272137715](https://de.wikipedia.org/w/index.php?oldid=272137715) – last modified 2026-11-13, retrieved 2026-11-15 – CC BY-SA 4.0 – `text049.txt`
50. **Grundgesetz für die Bundesrepublik Deutschland** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Grundgesetz_f%C3%BCr_die_Bundesrepublik_Deutschland> – revision [271976864](https://de.wikipedia.org/w/index.php?oldid=271976864) – last modified 2026-11-08, retrieved 2026-11-15 – CC BY-SA 4.0 – `text050.txt`
51. **Computer** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Computer> – revision [267555454](https://de.wikipedia.org/w/index.php?oldid=267555454) – last modified 2026-05-31, retrieved 2026-11-15 – CC BY-SA 4.0 – `text051.txt`
52. **Universität** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Universit%C3%A4t> – revision [271775259](https://de.wikipedia.org/w/index.php?oldid=271775259) – last modified 2026-10-31, retrieved 2026-11-15 – CC BY-SA 4.0 – `text052.txt`
53. **Dresden** (Städte) – <https://de.wikipedia.org/wiki/Dresden> – revision [272052822](https://de.wikipedia.org/w/index.php?oldid=272052822) – last modified 2026-11-10, retrieved 2026-11-15 – CC BY-SA 4.0 – `text053.txt`
54. **Mosel** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Mosel> – revision [270345237](https://de.wikipedia.org/w/index.php?oldid=270345237) – last modified 2026-09-09, retrieved 2026-11-15 – CC BY-SA 4.0 – `text054.txt`
55. **Martin Luther** (Personen) – <https://de.wikipedia.org/wiki/Martin_Luther> – revision [271338675](https://de.wikipedia.org/w/index.php?oldid=271338675) – last modified 2026-10-14, retrieved 2026-11-15 – CC BY-SA 4.0 – `text055.txt`
56. **Deutscher Bundestag** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Deutscher_Bundestag> – revision [270761080](https://de.wikipedia.org/w/index.php?oldid=270761080) – last modified 2026-09-24, retrieved 2026-11-15 – CC BY-SA 4.0 – `text056.txt`
57. **Bier** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Bier> – revision [272136480](https://de.wikipedia.org/w/index.php?oldid=272136480) – last modified 2026-11-13, retrieved 2026-11-15 – CC BY-SA 4.0 – `text057.txt`
58. **Internet** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Internet> – revision [271905371](https://de.wikipedia.org/w/index.php?oldid=271905371) – last modified 2026-11-05, retrieved 2026-11-15 – CC BY-SA 4.0 – `text058.txt`
59. **Demokratie** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Demokratie> – revision [271879280](https://de.wikipedia.org/w/index.php?oldid=271879280) – last modified 2026-11-04, retrieved 2026-11-15 – CC BY-SA 4.0 – `text059.txt`
60. **Nürnberg** (Städte) – <https://de.wikipedia.org/wiki/N%C3%BCrnberg> – revision [272014742](https://de.wikipedia.org/w/index.php?oldid=272014742) – last modified 2026-11-09, retrieved 2026-11-15 – CC BY-SA 4.0 – `text060.txt`
61. **Lüneburger Heide** (Natur und Geografie) – <https://de.wikipedia.org/wiki/L%C3%BCneburger_Heide> – revision [271998232](https://de.wikipedia.org/w/index.php?oldid=271998232) – last modified 2026-11-08, retrieved 2026-11-15 – CC BY-SA 4.0 – `text061.txt`
62. **Otto von Bismarck** (Personen) – <https://de.wikipedia.org/wiki/Otto_von_Bismarck> – revision [272152294](https://de.wikipedia.org/w/index.php?oldid=272152294) – last modified 2026-11-14, retrieved 2026-11-15 – CC BY-SA 4.0 – `text062.txt`
63. **Deutscher Fußball-Bund** (Unternehmen und Organisationen) – <https://de.wikipedia.org/wiki/Deutscher_Fu%C3%9Fball-Bund> – revision [272138072](https://de.wikipedia.org/w/index.php?oldid=272138072) – last modified 2026-11-13, retrieved 2026-11-15 – CC BY-SA 4.0 – `text063.txt`
64. **Europäische Union** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Europ%C3%A4ische_Union> – revision [272042367](https://de.wikipedia.org/w/index.php?oldid=272042367) – last modified 2026-11-10, retrieved 2026-11-15 – CC BY-SA 4.0 – `text064.txt`
65. **Fußball** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Fu%C3%9Fball> – revision [272184027](https://de.wikipedia.org/w/index.php?oldid=272184027) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text065.txt`
66. **Föderalismus** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/F%C3%B6deralismus> – revision [270384125](https://de.wikipedia.org/w/index.php?oldid=270384125) – last modified 2026-09-10, retrieved 2026-11-15 – CC BY-SA 4.0 – `text066.txt`
67. **Alpen** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Alpen> – revision [271907834](https://de.wikipedia.org/w/index.php?oldid=271907834) – last modified 2026-11-05, retrieved 2026-11-15 – CC BY-SA 4.0 – `text067.txt`
68. **Marie Curie** (Personen) – <https://de.wikipedia.org/wiki/Marie_Curie> – revision [271831298](https://de.wikipedia.org/w/index.php?oldid=271831298) – last modified 2026-11-02, retrieved 2026-11-15 – CC BY-SA 4.0 – `text068.txt`
69. **Bundesliga** (Unternehmen und Organisationen) – <https://de.wikipedia.org/wiki/Bundesliga> – revision [269894091](https://de.wikipedia.org/w/index.php?oldid=269894091) – last modified 2026-08-22, retrieved 2026-11-15 – CC BY-SA 4.0 – `text069.txt`
70. **Hanse** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Hanse> – revision [272059212](https://de.wikipedia.org/w/index.php?oldid=272059212) – last modified 2026-11-10, retrieved 2026-11-15 – CC BY-SA 4.0 – `text070.txt`
71. **Märchen** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/M%C3%A4rchen> – revision [271788085](https://de.wikipedia.org/w/index.php?oldid=271788085) – last modified 2026-10-31, retrieved 2026-11-15 – CC BY-SA 4.0 – `text071.txt`
72. **Energiewende** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Energiewende> – revision [271378780](https://de.wikipedia.org/w/index.php?oldid=271378780) – last modified 2026-10-16, retrieved 2026-11-15 – CC BY-SA 4.0 – `text072.txt`
73. **Sozialstaat** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Sozialstaat> – revision [270014501](https://de.wikipedia.org/w/index.php?oldid=270014501) – last modified 2026-08-27, retrieved 2026-11-15 – CC BY-SA 4.0 – `text073.txt`
74. **Hannover** (Städte) – <https://de.wikipedia.org/wiki/Hannover> – revision [272167743](https://de.wikipedia.org/w/index.php?oldid=272167743) – last modified 2026-11-14, retrieved 2026-11-15 – CC BY-SA 4.0 – `text074.txt`
75. **Nordsee** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Nordsee> – revision [270808419](https://de.wikipedia.org/w/index.php?oldid=270808419) – last modified 2026-09-26, retrieved 2026-11-15 – CC BY-SA 4.0 – `text075.txt`
76. **Franz Kafka** (Personen) – <https://de.wikipedia.org/wiki/Franz_Kafka> – revision [269458424](https://de.wikipedia.org/w/index.php?oldid=269458424) – last modified 2026-08-07, retrieved 2026-11-15 – CC BY-SA 4.0 – `text076.txt`
77. **Max-Planck-Gesellschaft** (Unternehmen und Organisationen) – <https://de.wikipedia.org/wiki/Max-Planck-Gesellschaft> – revision [271379409](https://de.wikipedia.org/w/index.php?oldid=271379409) – last modified 2026-10-16, retrieved 2026-11-15 – CC BY-SA 4.0 – `text077.txt`
78. **Industrielle Revolution** (Geschichte und Politik) – <https://de.wikipedia.org/wiki/Industrielle_Revolution> – revision [270016856](https://de.wikipedia.org/w/index.php?oldid=270016856) – last modified 2026-08-27, retrieved 2026-11-15 – CC BY-SA 4.0 – `text078.txt`
79. **Röntgenstrahlung** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/R%C3%B6ntgenstrahlung> – revision [272176563](https://de.wikipedia.org/w/index.php?oldid=272176563) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text079.txt`
80. **Heidelberg** (Städte) – <https://de.wikipedia.org/wiki/Heidelberg> – revision [272007327](https://de.wikipedia.org/w/index.php?oldid=272007327) – last modified 2026-11-09, retrieved 2026-11-15 – CC BY-SA 4.0 – `text080.txt`
81. **Thomas Mann** (Personen) – <https://de.wikipedia.org/wiki/Thomas_Mann> – revision [271732445](https://de.wikipedia.org/w/index.php?oldid=271732445) – last modified 2026-10-29, retrieved 2026-11-15 – CC BY-SA 4.0 – `text081.txt`
82. **Bauhaus** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Bauhaus> – revision [272113782](https://de.wikipedia.org/w/index.php?oldid=272113782) – last modified 2026-11-12, retrieved 2026-11-15 – CC BY-SA 4.0 – `text082.txt`
83. **Impfung** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Impfung> – revision [271858231](https://de.wikipedia.org/w/index.php?oldid=271858231) – last modified 2026-11-03, retrieved 2026-11-15 – CC BY-SA 4.0 – `text083.txt`
84. **Weimar** (Städte) – <https://de.wikipedia.org/wiki/Weimar> – revision [271608768](https://de.wikipedia.org/w/index.php?oldid=271608768) – last modified 2026-10-25, retrieved 2026-11-15 – CC BY-SA 4.0 – `text084.txt`
85. **Angela Merkel** (Personen) – <https://de.wikipedia.org/wiki/Angela_Merkel> – revision [271822060](https://de.wikipedia.org/w/index.php?oldid=271822060) – last modified 2026-11-02, retrieved 2026-11-15 – CC BY-SA 4.0 – `text085.txt`
86. **Deutsche Telekom** (Unternehmen und Organisationen) – <https://de.wikipedia.org/wiki/Deutsche_Telekom> – revision [272136953](https://de.wikipedia.org/w/index.php?oldid=272136953) – last modified 2026-11-13, retrieved 2026-11-15 – CC BY-SA 4.0 – `text086.txt`
87. **Dampfmaschine** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Dampfmaschine> – revision [271489554](https://de.wikipedia.org/w/index.php?oldid=271489554) – last modified 2026-10-20, retrieved 2026-11-15 – CC BY-SA 4.0 – `text087.txt`
88. **Niederdeutsche Sprache** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Niederdeutsche_Sprache> – revision [272182141](https://de.wikipedia.org/w/index.php?oldid=272182141) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text088.txt`
89. **Wien** (Städte) – <https://de.wikipedia.org/wiki/Wien> – revision [270892903](https://de.wikipedia.org/w/index.php?oldid=270892903) – last modified 2026-09-29, retrieved 2026-11-15 – CC BY-SA 4.0 – `text089.txt`
90. **Konrad Adenauer** (Personen) – <https://de.wikipedia.org/wiki/Konrad_Adenauer> – revision [271861562](https://de.wikipedia.org/w/index.php?oldid=271861562) – last modified 2026-11-03, retrieved 2026-11-15 – CC BY-SA 4.0 – `text090.txt`
91. **Porsche** (Unternehmen und Organisationen) – <https://de.wikipedia.org/wiki/Porsche> – revision [271381110](https://de.wikipedia.org/w/index.php?oldid=271381110) – last modified 2026-10-16, retrieved 2026-11-15 – CC BY-SA 4.0 – `text091.txt`
92. **Fernsehen** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Fernsehen> – revision [270791965](https://de.wikipedia.org/w/index.php?oldid=270791965) – last modified 2026-09-25, retrieved 2026-11-15 – CC BY-SA 4.0 – `text092.txt`
93. **Geschlechtergerechte Sprache** (Sprache und Gesellschaft) – <https://de.wikipedia.org/wiki/Geschlechtergerechte_Sprache> – revision [272101371](https://de.wikipedia.org/w/index.php?oldid=272101371) – last modified 2026-11-12, retrieved 2026-11-15 – CC BY-SA 4.0 – `text093.txt`
94. **Zürich** (Städte) – <https://de.wikipedia.org/wiki/Z%C3%BCrich> – revision [272191887](https://de.wikipedia.org/w/index.php?oldid=272191887) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text094.txt`
95. **Thüringer Wald** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Th%C3%BCringer_Wald> – revision [269852835](https://de.wikipedia.org/w/index.php?oldid=269852835) – last modified 2026-08-21, retrieved 2026-11-15 – CC BY-SA 4.0 – `text095.txt`
96. **Hildegard von Bingen** (Personen) – <https://de.wikipedia.org/wiki/Hildegard_von_Bingen> – revision [272171709](https://de.wikipedia.org/w/index.php?oldid=272171709) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text096.txt`
97. **Glühlampe** (Wissenschaft und Technik) – <https://de.wikipedia.org/wiki/Gl%C3%BChlampe> – revision [271229327](https://de.wikipedia.org/w/index.php?oldid=271229327) – last modified 2026-10-11, retrieved 2026-11-15 – CC BY-SA 4.0 – `text097.txt`
98. **Ostsee** (Natur und Geografie) – <https://de.wikipedia.org/wiki/Ostsee> – revision [272130279](https://de.wikipedia.org/w/index.php?oldid=272130279) – last modified 2026-11-13, retrieved 2026-11-15 – CC BY-SA 4.0 – `text098.txt`
99. **Alexander von Humboldt** (Personen) – <https://de.wikipedia.org/wiki/Alexander_von_Humboldt> – revision [271991545](https://de.wikipedia.org/w/index.php?oldid=271991545) – last modified 2026-11-08, retrieved 2026-11-15 – CC BY-SA 4.0 – `text099.txt`
100. **Kleingarten** (Kultur und Alltag) – <https://de.wikipedia.org/wiki/Kleingarten> – revision [272188181](https://de.wikipedia.org/w/index.php?oldid=272188181) – last modified 2026-11-15, retrieved 2026-11-15 – CC BY-SA 4.0 – `text100.txt`

## 3 · Corpus statistics  *(Darsi)*
Definitions used everywhere below and in the table row:
- **Tokens:** every spaCy token, including punctuation and numbers.
- **Word tokens:** tokens made only of letters (`is_alpha`), lower-cased.
- **Unique words:** distinct word tokens (types).
- **Stop-word share:** word tokens in spaCy's German stop-word list, divided by all word tokens.

In [8]:
SW = nlp.Defaults.stop_words
docs_raw = stats.parse(nlp, raw)
st = stats.compute(docs_raw, SW)
display(st.overview())

,value
measure,
Texts,100
"Tokens (all, incl. punctuation/numbers)",24040
Word tokens (alphabetic),19965
"Unique words (types, lower-cased)",6124
Unique lemmas,5145
Type-token ratio,0.307
Sentences,1123
Avg. word tokens per text,199.7
Avg. tokens per sentence,21.4


In [9]:
print('Top 10 words (all) and top 10 content words (stop words removed):')
display(stats.top_table(st, 10, SW))

Top 10 words (all) and top 10 content words (stop words removed):


,word,count,share,stop_word,content_word,content_count
rank,,,,,,
1,der,963,4.8%,True,stadt,72
2,und,810,4.1%,True,deutschen,68
3,die,728,3.6%,True,deutschland,50
4,in,408,2.0%,True,deutsche,43
5,von,296,1.5%,True,millionen,36
6,des,264,1.3%,True,deutschlands,34
7,ist,258,1.3%,True,bezeichnet,34
8,mit,232,1.2%,True,einwohnern,31
9,im,232,1.2%,True,gilt,29


In [10]:
topic_of = dict(zip(sources.file, sources.topic))
display(stats.per_topic(docs_raw, topic_of, SW))

,texts,word_tokens,unique_words,ttr,top_content_words
topic,,,,,
Geschichte und Politik,12,3011,1263,0.419,"deutschen, reich, deutsche, krieg, ddr"
Kultur und Alltag,11,1941,976,0.503,"karneval, märchen, bier, deutschland, brezel"
Natur und Geografie,13,2156,944,0.438,"liegt, harz, deutschland, alpen, meer"
Personen,16,3482,1468,0.422,"gilt, deutscher, werke, beethoven, mozart"
Sprache und Gesellschaft,11,2053,961,0.468,"sprache, universität, deutsche, deutsch, umlaut"
Städte,15,3501,1224,0.350,"stadt, einwohnern, millionen, deutschlands, sitz"
Unternehmen und Organisationen,8,1118,510,0.456,"deutsche, unternehmen, deutschen, porsche, siemens"
Wissenschaft und Technik,14,2703,1258,0.465,"bezeichnet, internet, computer, energiewende, relativitätstheorie"


In [11]:
n_stop_top10 = sum(w in SW for w, _ in st.top(10))
pt = stats.per_topic(docs_raw, topic_of, SW)
ttr_city = pt.ttr['Städte']; others = pt.ttr.drop('Städte')
ttr_min_other, ttr_max_other = others.min(), others.max()
display(Markdown(
    f"**Reading the numbers.** {n_stop_top10} of the 10 most frequent words are spaCy stop words "
    f"(articles *der/die/des*, conjunction *und*, prepositions *in/von/mit/im*). Function words make up "
    f"{st.stopword_share:.1%} of all word tokens. Once they are removed, the top words show the "
    f"encyclopedic topics: *{', '.join(w for w, _ in st.top(5, content_only=True))}*. "
    f"Several of them are inflected forms of the same lemma (*deutsche/deutschen*, *deutschland/deutschlands*). "
    f"That is German case morphology at work, and a reason to count lemmas as well: "
    f"{st.unique_words} word types collapse to {st.unique_lemmas} lemmas. "
    f"The type-token ratio of {st.ttr:.2f} is high for a corpus of {st.word_tokens:,} word tokens, because "
    f"every text is on a different topic and compounds rarely repeat. The per-topic table shows each topic's own vocabulary "
    f"(*stadt, einwohnern* for cities; *werke, gilt* for people). Cities have the lowest type-token ratio "
    f"({ttr_city:.2f}, against {ttr_min_other:.2f}–{ttr_max_other:.2f} for the other topics). City intros follow a fixed "
    f"pattern (population, area, administrative seat), so they repeat the same words."))

**Reading the numbers.** 10 of the 10 most frequent words are spaCy stop words (articles *der/die/des*, conjunction *und*, prepositions *in/von/mit/im*). Function words make up 47.2% of all word tokens. Once they are removed, the top words show the encyclopedic topics: *stadt, deutschen, deutschland, deutsche, millionen*. Several of them are inflected forms of the same lemma (*deutsche/deutschen*, *deutschland/deutschlands*). That is German case morphology at work, and a reason to count lemmas as well: 6124 word types collapse to 5145 lemmas. The type-token ratio of 0.31 is high for a corpus of 19,965 word tokens, because every text is on a different topic and compounds rarely repeat. The per-topic table shows each topic's own vocabulary (*stadt, einwohnern* for cities; *werke, gilt* for people). Cities have the lowest type-token ratio (0.35, against 0.42–0.50 for the other topics). City intros follow a fixed pattern (population, area, administrative seat), so they repeat the same words.

## 4 · Quality problems  *(Geri & Darsi)*
### 4.1 Automatic checks on the raw corpus

In [12]:
rep_raw = quality.pattern_report(raw)
display(rep_raw)
print('Repeated sentences (>= 5 words):', quality.repeated_sentences(raw) or 'none')
print('Possibly English sentences     :', quality.english_sentences(raw) or 'none')
a, b, j = quality.max_similarity(raw)
print(f'Near-duplicate texts (Jaccard of 5-word shingles >= 0.3): {quality.near_duplicates(raw) or "none"}'
      f'  (most similar pair: {a} / {b}, J = {j})')

,check,files_affected,occurrences,example,files
0,mojibake (broken UTF-8),0,0,,
1,HTML tags,0,0,,
2,HTML entities,0,0,,
3,"wiki markup (==, [[, {{)",0,0,,
4,IPA transcription,12,19,text001.txt: …Berlin ([bɛr'li:n] ) ist die Hauptstadt und…,"text001.txt, text007.txt, text008.txt, text015.txt, text024.txt, text041.txt, text047.txt, text051.txt …"
5,empty brackets,1,1,text027.txt: …Frankfurt am Main () ist mit 760.656 Einwohne…,text027.txt
6,"dangling label / space before , ;",5,8,"text002.txt: …inisch Rhenus, französisch , italienisch Reno, rätoro…","text002.txt, text015.txt, text041.txt, text047.txt, text054.txt"
7,multiple spaces,12,13,"text002.txt: …nisch Reno, rätoromanisch auch surselvisch Rein, su…","text002.txt, text007.txt, text008.txt, text047.txt, text052.txt, text066.txt, text074.txt, text077.txt …"
8,non-Latin script,6,8,"text014.txt: …dialectos, altgriechisch διάλεκτος diálektos, deutsch ‚Gesp…","text014.txt, text015.txt, text031.txt, text046.txt, text059.txt, text076.txt"
9,boilerplate words,0,0,,


Repeated sentences (>= 5 words): none
Possibly English sentences     : none


Near-duplicate texts (Jaccard of 5-word shingles >= 0.3): none  (most similar pair: text022.txt / text041.txt, J = 0.011)


What the checks show:
- **No HTML, wiki markup, mojibake or boilerplate.** The API's `explaintext` mode returns clean plain text. (The boilerplate check is case-sensitive, so an in-text "*(siehe auch …)*" is not flagged.)
- **IPA transcriptions** such as `[ˈdʏsl̩ˌdɔʁf]` appear in 12 texts, always in the first sentence (mostly cities). They are not German text, and spaCy tags them as `PROPN` or `X` (section 5, sentence 1).
- **Stripped templates leave holes.** The API removes some templates (pronunciation, foreign-language names) but keeps their labels. The result is empty brackets (`Frankfurt am Main () ist …`) and dangling labels (`französisch , italienisch Reno`). A space before a comma or semicolon never occurs in normal Wikipedia text, so it reliably signals these holes.
- **Multiple spaces** are also left where templates were removed.
- **Non-Latin script** occurs only in Greek etymologies such as *Photosynthese (altgriechisch φῶς …)* and in Kafka's Hebrew name. This is legitimate content, so we keep it.
- **No repeated, English or near-duplicate texts.** Each article covers a distinct topic.

### 4.2 Cleaning
`clean.clean_text()` applies named regex rules in two phases. First it removes artefacts until nothing more changes, then it tidies punctuation and whitespace. The order matters because the removal rules recognise dangling labels by the whitespace gap that tidying would delete. The rules are covered by unit tests with real examples from this corpus (`tests/test_clean.py`). The cleaned copy is written to `corpus_clean/`; the raw corpus stays untouched.

In [13]:
hits = clean.clean_corpus()
total = sum(hits.values(), Counter())
print(f'{sum(1 for h in hits.values() if h)} of {len(hits)} texts changed. Rule hits:')
display(pd.Series(dict(total.most_common()), name='hits').to_frame())
clean_texts = corpus.load(config.CORPUS_CLEAN)
for f in ['text041.txt', 'text054.txt']:
    print(f"\n{f}\n  raw  : {raw[f][:130]}…\n  clean: {clean_texts[f][:130]}…")

22 of 100 texts changed. Rule hits:


,hits
ipa_transcription,19
multiple_spaces,11
dangling_language_label,9
bracket_separator,8
empty_brackets,5
space_before_punct,5
double_separator,3
dangling_conjunction,2
dangling_pronunciation_label,2



text041.txt
  raw  : Düsseldorf ([ˈdʏsl̩ˌdɔʁf] , Düsseldorfer Platt Düsseldörp) ist die Landeshauptstadt Nordrhein-Westfalens und der Behördensitz des …
  clean: Düsseldorf (Düsseldorfer Platt Düsseldörp) ist die Landeshauptstadt Nordrhein-Westfalens und der Behördensitz des Regierungsbezirk…

text054.txt
  raw  : Die Mosel (französisch , luxemburgisch , lateinisch Mosella) ist mit 544 km Fließstrecke der zweitlängste Nebenfluss des Rheins na…
  clean: Die Mosel (lateinisch Mosella) ist mit 544 km Fließstrecke der zweitlängste Nebenfluss des Rheins nach der Maas. Der Fluss, dessen…


In [14]:
rep_clean = quality.pattern_report(clean_texts)
display(rep_raw[['check', 'files_affected', 'occurrences']].merge(
    rep_clean[['check', 'files_affected', 'occurrences']], on='check', suffixes=(' (raw)', ' (clean)')).set_index('check'))
st_clean = stats.compute(stats.parse(nlp, clean_texts), SW)
display(stats.compare(st, st_clean))

,files_affected (raw),occurrences (raw),files_affected (clean),occurrences (clean)
check,,,,
mojibake (broken UTF-8),0,0,0,0
HTML tags,0,0,0,0
HTML entities,0,0,0,0
"wiki markup (==, [[, {{)",0,0,0,0
IPA transcription,12,19,0,0
empty brackets,1,1,0,0
"dangling label / space before , ;",5,8,0,0
multiple spaces,12,13,0,0
non-Latin script,6,8,5,7


,raw,clean
measure,,
Texts,100,100
"Tokens (all, incl. punctuation/numbers)",24040,23940
Word tokens (alphabetic),19965,19941
"Unique words (types, lower-cased)",6124,6108
Unique lemmas,5145,5129
Type-token ratio,0.307,0.306
Sentences,1123,1117
Avg. word tokens per text,199.7,199.4
Avg. tokens per sentence,21.4,21.4


After cleaning, every targeted artefact count is zero. Only the deliberately kept Greek and Hebrew etymologies remain. The statistics hardly change (about 0.4% of tokens, mostly IPA fragments and stray punctuation), so the artefacts are rare. They are still worth removing: they sit in the first sentence of the most prominent articles, and there they break POS tags and parses (section 5).

## 5 · Three sentences through the pipeline  *(Blevis)*
The three sentences are **pinned** in `config.PINNED_SENTENCES` instead of auto-selected, so the written verdicts always match the output. A test (`tests/test_data_integrity.py`) fails if a rebuilt corpus no longer contains them. They were chosen to cover different difficulties:
1. Proper names, an IPA artefact and coordinated noun phrases (raw text).
2. A long compound and a past-tense verb that looks like an adjective.
3. A contraction (*im*), a fronted prepositional phrase and a subject placed after the verb.

In [15]:
def show(i, sentence):
    d = nlp(sentence)
    src = analysis.locate(sentence, raw)
    title = sources.set_index('file').title.get(src, '?')
    display(Markdown(f"**Sentence {i}** (from `{src}`, *{title}*):\n\n> {sentence}"))
    display(analysis.token_table(d, morph=True))
    paths = analysis.save_displacy(d, f'sentence{i}')
    display(HTML(paths['dep'].read_text(encoding='utf-8')))
    displacy.render(d, style='ent', jupyter=True)
    return d

s1 = show(1, config.PINNED_SENTENCES[0])

**Sentence 1** (from `text041.txt`, *Düsseldorf*):

> Düsseldorf ([ˈdʏsl̩ˌdɔʁf] , Düsseldorfer Platt Düsseldörp) ist die Landeshauptstadt Nordrhein-Westfalens und der Behördensitz des Regierungsbezirks Düsseldorf.

,token,lemma,pos,tag,morph,dep,head,ent
i,,,,,,,,
0,Düsseldorf,Düsseldorf,PROPN,NE,Case=Nom|Gender=Neut|Number=Sing,sb,ist,B-LOC
1,(,--,PUNCT,$(,,punct,[,
2,[,[,PROPN,NE,Case=Nom|Number=Sing,par,Düsseldorf,
3,ˈdʏsl̩ˌdɔʁf,ˈdʏsl̩ˌdɔʁf,PROPN,NE,Case=Gen|Number=Sing,punct,[,
4,],],PUNCT,$(,,punct,[,
5,",",--,PUNCT,"$,",,punct,Düsseldorf,
6,Düsseldorfer,Düsseldorfer,ADJ,ADJA,Degree=Pos,nk,Platt,B-MISC
7,Platt,Platt,NOUN,NN,Case=Nom|Gender=Masc|Number=Sing,cj,Düsseldorf,B-LOC
8,Düsseldörp,Düsseldörp,PROPN,NE,Case=Nom|Number=Sing,nk,Platt,I-LOC


**Sentence 1: verdict**

**Correct:**
- The clause structure is right. *Düsseldorf* is the subject (`sb`) of *ist* (ROOT), and *Landeshauptstadt* is the predicative (`pd`).
- *Nordrhein-Westfalens* and *Regierungsbezirks* are genitive attributes (`ag`). The morphologizer marks both `Case=Gen`, and the lemmatizer removes the genitive *-s* (*Nordrhein-Westfalen*, *Regierungsbezirk*).
- *und* coordinates the two predicate nouns (`cd`/`cj`).

**Incorrect:**
- *Behördensitz* ("seat of the authorities") is a common noun, but it is tagged `PROPN`/`NE` and even recognised as a **LOC entity**. German capitalises all nouns, so capitalisation is no clue to a name, and the model seems to guess from position (after *der*, inside a chain of place names).
- The IPA artefact is analysed as German: `[` becomes a `PROPN` with the relation `par`, and `ˈdʏsl̩ˌdɔʁf` becomes a `PROPN` with `Case=Gen`.
- The entity span *Platt Düsseldörp* is labelled LOC. The language name is *Düsseldorfer Platt* (MISC); only *Düsseldörp* is a place name (LOC).

In [16]:
print('Same sentence after cleaning (section 4):')
s1c = nlp(clean.clean_text(config.PINNED_SENTENCES[0]))
display(analysis.token_table(s1c)[['token', 'pos', 'dep', 'head', 'ent']])

Same sentence after cleaning (section 4):


,token,pos,dep,head,ent
i,,,,,
0,Düsseldorf,PROPN,sb,ist,B-LOC
1,(,PUNCT,punct,Platt,
2,Düsseldorfer,ADJ,nk,Platt,B-MISC
3,Platt,NOUN,par,Düsseldorf,B-LOC
4,Düsseldörp,PROPN,nk,Platt,I-LOC
5,),PUNCT,punct,Platt,
6,ist,AUX,ROOT,ist,
7,die,DET,nk,Landeshauptstadt,
8,Landeshauptstadt,NOUN,pd,ist,


With the IPA removed, the bracket becomes one well-formed parenthesis (`par` → *Düsseldorf*), and no IPA string is tagged as a proper noun. The *Behördensitz* error remains, so that one is a **model error**, while the IPA tokens were a **data error** that cleaning fixes.

In [17]:
s2 = show(2, config.PINNED_SENTENCES[1])

**Sentence 2** (from `text028.txt`, *Johann Sebastian Bach*):

> Seine Werke beeinflussten spätere Komponistengenerationen und inspirieren bis heute musikschaffende Künstler zu zahllosen Bearbeitungen.

,token,lemma,pos,tag,morph,dep,head,ent
i,,,,,,,,
0,Seine,sein,DET,PPOSAT,Case=Nom|Gender=Neut|Number=Plur|Poss=Yes|PronType=Prs,nk,Werke,
1,Werke,Werk,NOUN,NN,Case=Nom|Gender=Neut|Number=Plur,sb,beeinflussten,
2,beeinflussten,beeinflusst,ADJ,ADJA,Case=Acc|Degree=Pos|Gender=Fem|Number=Plur,ROOT,beeinflussten,
3,spätere,spät,ADJ,ADJA,Case=Acc|Degree=Cmp|Gender=Fem|Number=Plur,nk,Komponistengenerationen,
4,Komponistengenerationen,Komponistengeneratione,NOUN,NN,Case=Acc|Gender=Fem|Number=Plur,oa,beeinflussten,B-LOC
5,und,und,CCONJ,KON,,cd,beeinflussten,
6,inspirieren,inspirieren,VERB,VVFIN,Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin,cj,und,
7,bis,bis,ADP,APPR,,mo,musikschaffende,
8,heute,heute,ADV,ADV,,nk,bis,


**Sentence 2: verdict**

**Correct:**
- *Werke* is the plural subject (`sb`, lemma *Werk*), and *inspirieren* is tagged as a finite verb (`VVFIN`) with *Künstler* as its accusative object (`oa`).
- The two clauses are coordinated with *und*.
- Lemmas of regular forms are right: *zahllosen → zahllos*, *Bearbeitungen → Bearbeitung*, *spätere → spät* (comparative marked `Degree=Cmp`).

**Incorrect:**
- **The main verb is not recognised.** *beeinflussten* is the finite past tense of *beeinflussen*, but it is tagged `ADJ`/`ADJA` (attributive adjective) with the lemma *beeinflusst*. The form is ambiguous: *die beeinflussten Künstler* would be an adjective. Here it is the sentence's verb, and the parser even makes it ROOT while still calling it an adjective.
- **The compound is lemmatised wrongly:** *Komponistengenerationen → Komponistengeneratione* (should be *Komponistengeneration*). The rare compound is also tagged as a **LOC entity**.
- *Bearbeitungen* gets `Gender=Masc`; the correct gender is feminine (*die Bearbeitung*).
- *bis heute* is attached to the adjective *musikschaffende*, but it modifies the verb *inspirieren* ("inspire to this day").

In [18]:
s3 = show(3, config.PINNED_SENTENCES[2])

**Sentence 3** (from `text001.txt`, *Berlin*):

> Neben den Flüssen Spree, Havel und Dahme befinden sich im Stadtgebiet kleinere Fließgewässer sowie zahlreiche Seen und Wälder.

,token,lemma,pos,tag,morph,dep,head,ent
i,,,,,,,,
0,Neben,neben,ADP,APPR,,mnr,Fließgewässer,
1,den,der,DET,ART,Case=Dat|Definite=Def|Gender=Masc|Number=Plur|PronType=Art,nk,Flüssen,
2,Flüssen,Fluß,NOUN,NN,Case=Dat|Gender=Masc|Number=Plur,nk,Neben,
3,Spree,Spree,PROPN,NE,Case=Nom|Gender=Neut|Number=Sing,nk,Flüssen,B-LOC
4,",",--,PUNCT,"$,",,punct,Spree,
5,Havel,Havel,PROPN,NE,Case=Dat|Gender=Neut|Number=Sing,cj,Spree,B-LOC
6,und,und,CCONJ,KON,,cd,Havel,
7,Dahme,Dahme,PROPN,NE,Case=Dat|Gender=Fem|Number=Sing,cj,und,B-LOC
8,befinden,befinden,VERB,VVFIN,Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin,ROOT,befinden,


**Sentence 3: verdict**

**Correct:**
- The verb, its reflexive and the subject are found. *befinden* is ROOT, *sich* is its reflexive accusative object (`PRF`, `oa`, `Reflex=Yes`), and the subject *Fließgewässer*, placed after the verb, is `sb`.
- The contraction *im* is tagged `APPRART` with the lemma *in*, and *im Stadtgebiet* is a verb modifier (`mo`).
- Both coordinations are parsed: *Spree, Havel und Dahme* and *Fließgewässer sowie … Seen und Wälder*. The three rivers are LOC entities.

**Incorrect:**
- **Attachment of the fronted phrase:** *Neben den Flüssen …* is attached to the noun *Fließgewässer* as a post-nominal modifier (`mnr`). In German the position before the finite verb (*Vorfeld*) holds exactly one constituent of the clause, so this PP is a modifier of *befinden* (`mo`). A noun that comes after the verb cannot own it.
- **Lemmas:** *Fließgewässer → Fließgewäss* (the plural is identical to the singular, but the lemmatizer cuts *-er* as if it were an ending). *Flüssen → Fluß* uses the spelling from before the 1996 reform (correct today: *Fluss*).
- *Spree* gets `Case=Nom`, but as an apposition to *den Flüssen* it is dative, like *Havel* and *Dahme*.

## 6 · German-specific behaviour and our table row  *(Darsi)*

In [19]:
print('Probes: compounds, contractions, ß, hyphenated names, separable verb, abbreviations')
display(analysis.probe_table(nlp))

Probes: compounds, contractions, ß, hyphenated names, separable verb, abbreviations

,input,n_tokens,tokens,lemmas,pos
0,Kraftfahrzeug-Haftpflichtversicherung,1,Kraftfahrzeug-Haftpflichtversicherung,Kraftfahrzeug-Haftpflichtversicherung,NOUN
1,Donaudampfschifffahrtsgesellschaft,1,Donaudampfschifffahrtsgesellschaft,Donaudampfschifffahrtsgesellschaft,NOUN
2,zum Bahnhof,2,zum | Bahnhof,zu | Bahnhof,ADP | NOUN
3,im Haus,2,im | Haus,in | Haus,ADP | NOUN
4,vom Rhein,2,vom | Rhein,von | Rhein,ADP | PROPN
5,Straße,1,Straße,Straße,NOUN
6,Müller-Lüdenscheidt,1,Müller-Lüdenscheidt,Müller-Lüdenscheidt,PROPN
7,Er ruft mich morgen an.,6,Er | ruft | mich | morgen | an | .,er | rufen | mich | morgen | an | --,PRON | VERB | PRON | ADV | ADP | PUNCT
8,z. B. am 3. Oktober 1990,6,z. | B. | am | 3. | Oktober | 1990,z. | B. | an | 3. | Oktober | 1990,ADP | NOUN | ADP | ADJ | NOUN | NUM


In [20]:
print('Longest words in the corpus (all single tokens):')
display(analysis.longest_words(docs_raw, 10))
print('Preposition + article contractions in the corpus:')
display(analysis.contraction_lemmas(docs_raw))

Longest words in the corpus (all single tokens):


,word,length,lemma,pos
0,Eisenbahninfrastrukturunternehmen,33,Eisenbahninfrastrukturunternehmen,NOUN
1,Telekommunikationsunternehmen,29,Telekommunikationsunternehmen,NOUN
2,Eisenbahnverkehrsunternehmen,28,Eisenbahnverkehrsunternehmen,NOUN
3,Fußballnationalmannschaften,27,Fußballnationalmannschafte,NOUN
4,kontinentalwestgermanische,26,kontinentalwestgermanisch,ADJ
5,geisteswissenschaftlichen,25,geisteswissenschaftlich,ADJ
6,Mehrheitsgesellschafterin,25,Mehrheitsgesellschafterin,NOUN
7,Kraftfahrzeugherstellern,24,Kraftfahrzeughersteller,NOUN
8,Rechtschreibwörterbücher,24,Rechtschreibwörterbücher,NOUN
9,Gesellschaftstheoretiker,24,Gesellschaftstheoretiker,NOUN


Preposition + article contractions in the corpus:


,form,count,lemma/tag (top 3)
0,am,59,"{'an/APPRART': 53, 'am/PTKA': 6}"
1,im,232,{'in/APPRART': 232}
2,zum,63,{'zu/APPRART': 63}
3,zur,69,{'zu/APPRART': 69}
4,vom,43,{'von/APPRART': 43}
5,beim,10,{'bei/APPRART': 10}
6,ins,6,{'in/APPRART': 6}


Observations:
- **Compounds are never split.** *Donaudampfschifffahrtsgesellschaft* and corpus words such as *Eisenbahninfrastrukturunternehmen* are single tokens with the whole word as lemma. spaCy does no decompounding, which explains the high type-token ratio: every new compound is a new type.
- **Hyphenated compounds and names** (*Kraftfahrzeug-Haftpflichtversicherung*, *Müller-Lüdenscheidt*, *Nordrhein-Westfalen*) also stay one token.
- **Contractions stay one token** (*zum*, *im*, *vom* are tagged `APPRART`). They are not split into *zu dem* etc., but the lemma is the bare preposition (*zu*, *in*, *von*). The article information survives only in the morphology (`Case=Dat|Gender=…`). *am* is split by function: `an/APPRART` in *am Rhein*, but `am/PTKA` (superlative particle) in all 6 cases of *am meisten* / *am dichtesten* in the corpus. That reading is correct.
- **Separable verbs are not rejoined.** In *Er ruft mich morgen an* the tagger correctly marks *an* as a separated verb particle (`PTKVZ`), but the lemma of *ruft* is *rufen*, so *anrufen* never appears as a lemma. Counts by lemma therefore mix *rufen* and *anrufen*.
- **Abbreviations and ordinals:** *z. B.* becomes two tokens (*z.* and *B.*), and *3.* is recognised as an ordinal (`ADJ`). Neither is mistaken for a sentence end.
- **Umlauts and ß** are kept in tokens and lemmas (*Straße*, *Fließgewässer*). The UTF-8 check in section 1 found no broken characters.

In [21]:
row = pd.DataFrame([{
    'Language': 'German',
    'Source': 'de.wikipedia.org (article introductions, 8 topics)',
    'Licence': config.LICENCE,
    'Texts': st.texts,
    'Tokens': st.tokens,
    'Word tokens': st.word_tokens,
    'Unique words': st.unique_words,
    'Problems found': quality.summary(raw) + ' (all but the legitimate non-Latin etymologies removed in corpus_clean/)',
}])
display(row.T.rename(columns={0: 'Group 2'}))

,Group 2
Language,German
Source,"de.wikipedia.org (article introductions, 8 topics)"
Licence,CC BY-SA 4.0
Texts,100
Tokens,24040
Word tokens,19965
Unique words,6124
Problems found,"IPA transcription: 12 files; empty brackets: 1 file; dangling label / space before , ;: 5 files; multiple spaces: 12..."


**1. Problems and causes.** All problems come from how the API turns wiki markup into plain text. Templates for pronunciation and foreign-language names are removed, but their labels and brackets stay, which leaves IPA strings, empty brackets, dangling labels and double spaces. There is no HTML or wiki markup, and no encoding damage.

**2. Long words, accents, contractions.** Compounds and hyphenated words stay single tokens and are not decompounded. Lemmatisation of rare compounds is unreliable (*Komponistengeneratione*, *Fließgewäss*). Umlauts and ß are handled correctly. Contractions stay one token with the bare preposition as lemma. Separable verb prefixes are not reattached.

**3. What to change before real use.** Use `corpus_clean/` (IPA, empty brackets and dangling labels removed). Count lemmas instead of surface forms for frequency work. Consider a decompounding step (e.g. a compound splitter) if word-level statistics matter. Hand-check a sample of PROPN tags and entities, because capitalised common nouns are the most frequent tagging error.

## 7 · Discussion  *(Geri)*

In [22]:
display(Markdown(
    f"**Is the corpus suitable?** For a first corpus, yes. Wikipedia introductions are free to reuse, consistently "
    f"encoded, and full of named entities, umlauts, ß and long compounds. They also come with exact revision metadata, so "
    f"the corpus can be reproduced. With {st.texts} texts in 8 balanced topics ({st.word_tokens:,} word tokens) it "
    f"already meets the week-3 target of about 100 texts. However, it is **not representative of everyday German**. The register is "
    f"formal and encyclopedic, with long nominal sentences ({st.tokens_per_sentence:.0f} tokens on average), and the topics "
    f"are our own choice, so *Stadt*, *Deutschland* and *deutsch-* are over-represented."))

**Is the corpus suitable?** For a first corpus, yes. Wikipedia introductions are free to reuse, consistently encoded, and full of named entities, umlauts, ß and long compounds. They also come with exact revision metadata, so the corpus can be reproduced. With 100 texts in 8 balanced topics (19,965 word tokens) it already meets the week-3 target of about 100 texts. However, it is **not representative of everyday German**. The register is formal and encyclopedic, with long nominal sentences (21 tokens on average), and the topics are our own choice, so *Stadt*, *Deutschland* and *deutsch-* are over-represented.

**Data problems.** The artefacts (IPA, empty brackets, dangling language labels) are rare, but they sit in the first sentence of the most prominent articles. There spaCy treats them as normal tokens; in the Düsseldorf sentence an IPA string became a genitive proper noun. They are systematic, so a small set of tested rules removes all of them (section 4). Cleaning rules can also do damage, so we kept them conservative: Greek etymologies and editorial brackets are left untouched.

**spaCy errors (section 5).** The model got the core of every sentence right: subjects, predicatives, genitive attributes, coordination, reflexives and contractions. The errors fall into three groups:
1. **Capitalisation:** common nouns tagged `PROPN`/LOC (*Behördensitz*, *Komponistengenerationen*).
2. **Ambiguous forms:** a finite past-tense verb tagged as an adjective (*beeinflussten*).
3. **Structure and lemmas:** a fronted PP attached to a noun, `bis heute` attached to the wrong head, and wrong lemmas for rare compounds and plurals that equal the singular (*Fließgewäss*), plus an old-spelling lemma (*Fluß*).

The cleaning experiment separates the two error sources. Removing the IPA fixed the data errors; the model errors stayed.

**German-specific behaviour.** No decompounding, so the vocabulary is large and sparse (see the type-token ratio). Contractions are kept whole but lemmatised to the preposition. Separable verbs are not rejoined. Umlauts and ß are handled correctly.

**What we would change before real use.**
- Add other text types (news, forum posts, fiction) for a balanced register.
- Work on `corpus_clean/` and use lemmas for frequency counts.
- Keep the per-text revision and licence metadata.
- Annotate a random sample by hand to measure the model's error rate, instead of relying on three sentences.

## Deliverable checklist (lab sheet 6.7)
| Item | Where |
|---|---|
| spaCy version printout and output for one sentence | §0 |
| displaCy images: dependency and entities | §0, §5 (also saved to `figures/`) |
| Three sentences, one correct and one incorrect result each | §5 |
| Corpus statistics (6.4) and our row of the table | §3, §6 |
| Source list with licences | §2, `sources.csv` |
| Notebook saved after *Restart Kernel and Run All Cells*, no errors | whole notebook (`python -m sprache notebook`) |